# Hybrid Retrieval with Pinecone + LangChain (BM25 + Dense + Rerank)


In [123]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

DATA_DIR = Path("data/fiqa")
INDEX_DIR = Path("indexes")
INDEX_DIR.mkdir(exist_ok=True)

assert os.getenv("PINECONE_API_KEY")
print("Pinecone API key loaded.")

Pinecone API key loaded.


In [124]:
from datasets import load_dataset

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "corpus.parquet").exists():
    corpus_ds = load_dataset("BeIR/fiqa", "corpus", split="corpus")
    queries_ds = load_dataset("BeIR/fiqa", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/fiqa-qrels", split="test")

    corpus_ds.to_parquet(DATA_DIR / "corpus.parquet")
    queries_ds.to_parquet(DATA_DIR / "queries.parquet")
    qrels_ds.to_parquet(DATA_DIR / "qrels.parquet")
    print("Downloaded and cached FiQA as parquet.")
else:
    print("FiQA already cached locally.")

FiQA already cached locally.


In [125]:
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
qrels = pd.read_parquet(DATA_DIR / "qrels.parquet")

In [40]:
corpus.head()

,_id,text
0,3,I'm not saying I don't like the idea of on-the...
1,31,So nothing preventing false ratings besides ad...
2,56,You can never use a health FSA for individual ...
3,59,Samsung created the LCD and other flat screen ...
4,63,Here are the SEC requirements: The federal sec...


In [130]:
corpus["text"] = corpus["text"].astype(str).str.strip()
before = len(corpus)
corpus = corpus[corpus["text"].str.len() > 0]
dropped = before - len(corpus)
if dropped:
    print(f"Dropped {dropped} empty/blank documents before indexing.")


Dropped 38 empty/blank documents before indexing.


In [131]:
doc_ids = corpus["_id"].tolist()
doc_texts = corpus["text"].tolist()

print(f"Corpus:  {len(corpus):>6} documents")
print(f"Queries: {len(queries):>6} total")
print(f"Qrels:   {len(qrels):>6} relevance judgments")
print("\nExample document:")
print(doc_texts[0][:300])

Corpus:   57600 documents
Queries:   6648 total
Qrels:     1706 relevance judgments

Example document:
I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange f


In [66]:
qrels.head()

,query-id,corpus-id,score
0,8,566392,1
1,8,65404,1
2,15,325273,1
3,18,88124,1
4,26,285255,1


In [132]:
qrels_small = qrels[qrels["query-id"].astype(str).isin(test_query_ids)]

In [133]:
qrels_small

,query-id,corpus-id,score
0,8,566392,1
1,8,65404,1
2,15,325273,1
3,18,88124,1
4,26,285255,1
...,...,...,...
104,813,436701,1
105,849,557186,1
106,852,245702,1
107,852,531288,1


In [134]:
test_query_ids = set(qrels["query-id"].astype(str).unique()[:50])
test_queries = queries[queries["_id"].astype(str).isin(test_query_ids)]
print(f"Test queries (with qrels): {len(test_queries)}")

example_query = test_queries.iloc[0]
print(f"\nExample query: {example_query['text']}")

relevant = qrels[qrels["query-id"].astype(str) == str(example_query["_id"])]
print(f"Relevant doc ids: {list(relevant['corpus-id'])}")

Test queries (with qrels): 50

Example query: Taxes due for hobbyist Group Buy
Relevant doc ids: [466718, 243503]


In [135]:
test_queries.iloc[4,2]

'Is it wise to have plenty of current accounts in different banks?'

In [144]:
test_queries.head()

,_id,title,text
6011,753,,Taxes due for hobbyist Group Buy
6024,570,,Employer options when setting up 401k for empl...
6026,594,,Should a retail trader bother about reading SE...
6049,603,,Will one’s education loan application be rejec...
6058,620,,Is it wise to have plenty of current accounts ...


In [136]:
# Keep every doc that's actually relevant to those 50 queries
relevant_doc_ids = set(qrels_small["corpus-id"].astype(str))
relevant_docs = corpus[corpus["_id"].astype(str).isin(relevant_doc_ids)]

In [137]:
n_target = 3000
extra_needed = n_target - len(relevant_docs)
other_docs = corpus[~corpus["_id"].astype(str).isin(relevant_doc_ids)].sample(
    extra_needed, random_state=42
)

corpus_small = pd.concat([relevant_docs, other_docs]).sample(frac=1, random_state=42)

corpus_small.to_parquet(DATA_DIR / "corpus_small.parquet")
qrels_small.to_parquet(DATA_DIR / "qrels_small.parquet")
queries[queries["_id"].astype(str).isin(test_query_ids)].to_parquet(
    DATA_DIR / "queries_small.parquet"
)

print(f"Corpus: {len(corpus_small)} docs ({len(relevant_docs)} relevant + {extra_needed} random)")
print(f"Queries: {len(test_query_ids)}")

Corpus: 3000 docs (109 relevant + 2891 random)
Queries: 50


This project deliberately skips chunking. Each row in the corpus (corpus["text"]) is treated as one whole document = one retrievable unit, passed directly into embedding/BM25/upsert as-is. That's a property of the FiQA benchmark — the docs are already short forum-post-sized chunks, so there's nothing to split further. 

In [139]:
corpus = pd.read_parquet(DATA_DIR / "corpus_small.parquet")
queries = pd.read_parquet(DATA_DIR / "queries_small.parquet")
qrels = pd.read_parquet(DATA_DIR / "qrels_small.parquet")

In [140]:
doc_ids = corpus["_id"].tolist()
doc_texts = corpus["text"].tolist()

print(f"Corpus:  {len(corpus):>6} documents")
print(f"Queries: {len(queries):>6} total")
print(f"Qrels:   {len(qrels):>6} relevance judgments")
print("\nExample document:")
print(doc_texts[0][:300])

Corpus:    3000 documents
Queries:     50 total
Qrels:      109 relevance judgments

Example document:
Read the Security Analysis.  I believe if you read it completely, you will have a real good chance of succeeding at making good money.  If you find the book hard to read just go through it and underline under the text as you read it.


## Sparse encoding with BM25


In [90]:
from pinecone_text.sparse import BM25Encoder

bm25 = BM25Encoder().default()
bm25.fit(doc_texts)

BM25_PARAMS_PATH = INDEX_DIR / "bm25_params.json"
bm25.dump(str(BM25_PARAMS_PATH))
print(f"Fit BM25 on {len(doc_texts)} docs, saved params to {BM25_PARAMS_PATH}")

100%|██████████| 3000/3000 [00:04<00:00, 689.31it/s]


Fit BM25 on 3000 docs, saved params to indexes\bm25_params.json


In [141]:
#checking
doc_sparse = bm25.encode_documents(doc_texts[0])
print(f"Doc sparse vector: {len(doc_sparse['indices'])} non-zero terms")

query_sparse = bm25.encode_queries("Where should I park my rainy-day fund?")
print(f"Query sparse vector: {len(query_sparse['indices'])} non-zero terms")

Doc sparse vector: 17 non-zero terms
Query sparse vector: 3 non-zero terms


## Dense embeddings via LangChain

We normalize so dotproduct == cosine similarity, which matters because Pinecone's hybrid index requires `metric="dotproduct"`.

In [60]:
from langchain_huggingface import HuggingFaceEmbeddings

DENSE_DIM = 384  # all-MiniLM-L6-v2 output size 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# quick check -- embed_query returns a plain python list of floats
sample_vec = embeddings.embed_query("test sentence")
print(f"Embedding dim: {len(sample_vec)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2153.83it/s]


Embedding dim: 384


## Create the Pinecone hybrid index


In [142]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
INDEX_NAME = "fiqa-hybrid-rag-small"

existing = [idx["name"] for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DENSE_DIM,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"Created index '{INDEX_NAME}'")
else:
    print(f"Index '{INDEX_NAME}' already exists, reusing it")

index = pc.Index(INDEX_NAME)

Index 'fiqa-hybrid-rag-small' already exists, reusing it


## Build the retriever and load the corpus

In [101]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    alpha=0.5,
    top_k=10,
)

BATCH = 200
if index.describe_index_stats()["total_vector_count"] < len(doc_ids):
    for i in tqdm(range(0, len(doc_ids), BATCH), desc="add_texts"):
        batch_ids = doc_ids[i : i + BATCH]
        batch_texts = doc_texts[i : i + BATCH]
        retriever.add_texts(
            texts=batch_texts,
            ids=batch_ids,
            metadatas=[{"doc_id": d} for d in batch_ids],
        )
    print(index.describe_index_stats())
else:
    print("Index already populated, skipping add_texts.")

add_texts: 100%|██████████| 15/15 [05:50<00:00, 23.34s/it]


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 3000}},
 'total_vector_count': 3000,
 'vector_type': 'dense'}


## Hybrid search: compare alpha values

In [117]:
sparse_retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.0, top_k=5)
dense_retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=1.0, top_k=5)
hybrid_retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.5, top_k=5)


def show(label, docs):
    print(f"\n{label}")
    for i, doc in enumerate(docs, 1):
        score = doc.metadata.get("score")
        doc_id = doc.metadata.get("doc_id")
        score_str = f"{score:.4f}" if score is not None else "  n/a "
        print(f"  {i}. [{score_str}] {doc_id}  {doc.page_content[:70]}")


query = 'Is it wise to have plenty of current accounts in different banks?'
print(f"Query: {query}")

show("Sparse only (alpha=0.0)", sparse_retriever.invoke(query))
show("Dense only (alpha=1.0)", dense_retriever.invoke(query))
show("Hybrid (alpha=0.5)", hybrid_retriever.invoke(query))

Query: Is it wise to have plenty of current accounts in different banks?

Sparse only (alpha=0.0)
  1. [0.3757] 510736  If your savings account linked to the mortgage account is an 100% offs
  2. [0.3099] 563025  In view of business, we have to book the entries. Business view, owner
  3. [0.2973] 456636  Current account offers a lot of benefits for sole proprietors. Think o
  4. [0.2915] 272328  It's never too early, but age 3 is when we started a piggy bank.  Age 
  5. [0.2415] 84630  It depends on how much equity you have in your home. Scenario 1: Your 

Dense only (alpha=1.0)
  1. [0.6717] 64556  If you're a sole proprietor there's no reason to have a separate busin
  2. [0.6031] 189303  Another thing to factor in are deals provided by banks. In general, ba
  3. [0.5711] 180673  I don't think there's any law against having lots of bank accounts. Bu
  4. [0.5623] 535566  For small amounts I wouldn't be too concerned.   There are two factors
  5. [0.5263] 58120  There are two answers 

## Rerank with ContextualCompressionRetriever

A bi-encoder embeds query and document separately. A **cross-encoder** feeds both into the same model together for a joint relevance score — slower, but catches subtleties a bi-encoder misses.



In [143]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_pinecone import PineconeRerank

hybrid_retriever_50 = PineconeHybridSearchRetriever(
    embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.5, top_k=50
)

reranker = PineconeRerank(model="bge-reranker-v2-m3", top_n=10)

reranked_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever_50,
)


def show_reranked(label, docs):
    print(f"\n{label}")
    for i, doc in enumerate(docs[:5], 1):
        score = doc.metadata.get("relevance_score", doc.metadata.get("score"))
        doc_id = doc.metadata.get("doc_id")
        print(f"  {i}. [{score:.4f}] {doc_id}  {doc.page_content[:70]}")


show_reranked("Hybrid (alpha=0.5) only", hybrid_retriever_50.invoke(query))
show_reranked("Hybrid + Pinecone rerank (bge-reranker-v2-m3)", reranked_retriever.invoke(query))


Hybrid (alpha=0.5) only
  1. [0.4200] 64556  If you're a sole proprietor there's no reason to have a separate busin
  2. [0.4172] 189303  Another thing to factor in are deals provided by banks. In general, ba
  3. [0.4118] 510736  If your savings account linked to the mortgage account is an 100% offs
  4. [0.3683] 142966  I believe different banks have slightly different experiences.  My wif
  5. [0.3480] 563025  In view of business, we have to book the entries. Business view, owner

Hybrid + Pinecone rerank (bge-reranker-v2-m3)
  1. [0.9361] 189303  Another thing to factor in are deals provided by banks. In general, ba
  2. [0.2053] 450586  I was just wondering, are banks in India federally insured? Yes the Ba
  3. [0.1947] 180673  I don't think there's any law against having lots of bank accounts. Bu
  4. [0.1438] 456636  Current account offers a lot of benefits for sole proprietors. Think o
  5. [0.1009] 64556  If you're a sole proprietor there's no reason to have a separate busin


## Evaluate with NDCG@10

In [149]:
import math
from collections import defaultdict

SAMPLE_SIZE = 50
SEED = 42


def ndcg_at_k(predicted_ids, relevant: dict, k: int = 10) -> float:
    dcg = sum(
        relevant.get(doc_id, 0) / math.log2(rank + 2)
        for rank, doc_id in enumerate(predicted_ids[:k])
    )
    ideal_rels = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0.0


qrels_map = defaultdict(dict)
for _, row in qrels.iterrows():
    qrels_map[str(row["query-id"])][str(row["corpus-id"])] = int(row["score"])

sample = test_queries.sample(n=SAMPLE_SIZE, random_state=SEED)
print(f"Evaluating on {len(sample)} queries")


def ids_from(docs):
    return [d.metadata["doc_id"] for d in docs]

Evaluating on 50 queries


In [150]:
eval_results = defaultdict(list)

sparse_r10 = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.0, top_k=10)
dense_r10 = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=1.0, top_k=10)
hybrid_r10 = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.5, top_k=10)

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Evaluating"):
    query_id = str(row["_id"])
    query_text = row["text"]
    relevant = qrels_map[query_id]

    eval_results["Sparse (BM25)"].append(ndcg_at_k(ids_from(sparse_r10.invoke(query_text)), relevant))
    eval_results["Dense (all-MiniLM-L6-v2)"].append(ndcg_at_k(ids_from(dense_r10.invoke(query_text)), relevant))
    eval_results["Hybrid (alpha=0.5)"].append(ndcg_at_k(ids_from(hybrid_r10.invoke(query_text)), relevant))
    eval_results["Hybrid + Rerank"].append(ndcg_at_k(ids_from(reranked_retriever.invoke(query_text)), relevant))

print(f"\nNDCG@10 x100 on FiQA ({len(sample)} sampled test queries)")
print("-" * 42)
for method, scores in eval_results.items():
    print(f"  {method:<26} {np.mean(scores) * 100:.2f}")

Evaluating: 100%|██████████| 50/50 [02:55<00:00,  3.52s/it]


NDCG@10 x100 on FiQA (50 sampled test queries)
------------------------------------------
  Sparse (BM25)              38.77
  Dense (all-MiniLM-L6-v2)   67.12
  Hybrid (alpha=0.5)         64.21
  Hybrid + Rerank            65.94
